# Fine Tune --> Embedding model

## Step 1: Library Imports

Importing the required libraries — `datasets` (to load data), `sentence_transformers` (embedding model, loss, trainer, training args) and `TripletEvaluator` (to evaluate the model's triplet accuracy).

In [ ]:
import json
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import TripletLoss # pyright: ignore[reportMissingImports]
from sentence_transformers.trainer import SentenceTransformerTrainer # pyright: ignore[reportMissingImports]
from sentence_transformers.training_args import SentenceTransformerTrainingArguments # pyright: ignore[reportMissingImports]
from datasets import load_dataset

from sentence_transformers.evaluation import TripletEvaluator




## Step 2: Loading the Dataset

Loading the `sentence-transformers/all-nli` dataset (triplet config) — each row contains an `anchor`, a `positive`, and a `negative` sentence. Taking 200 rows for training and a separate (unseen) 100 rows for evaluation, so the eval data never leaks into training.

In [171]:
# Load data
ds = load_dataset("sentence-transformers/all-nli", "triplet")
train = ds['train']
eval=ds['dev']



# Training ke liye alag rows
rows = train[:200]

# Eval ke liye COMPLETELY alag rows (jo training mein kabhi nahi gaye)
eval_rows = eval[:100]


## Step 3: Checking the Dataset Structure

Inspecting the structure of the loaded dataset (`ds`) — how many splits (train/dev/test) it has and how many rows/columns each split contains.

In [ ]:
# The loded data structure
ds

## Step 4: Converting Data into the Model's Format

Converting the raw rows into a HuggingFace `Dataset` object (with anchor/positive/negative columns) so the trainer can use it directly. Also creating a `TripletEvaluator` that will check the model's triplet accuracy on the separate eval rows after training.

In [ ]:
# Convert the data for model

train_dataset = Dataset.from_dict({
    "anchor": rows["anchor"],
    "positive": rows["positive"],
    "negative": rows["negative"],
})

evaluator = TripletEvaluator(
    anchors=eval_rows["anchor"],
    positives=eval_rows["positive"],
    negatives=eval_rows["negative"],
    name="triplet-eval-clean"
)

## Step 5: Setting Up the Base Model, Loss, and Training Arguments

Loading the `BAAI/bge-small-en-v1.5` base embedding model. We'll use `TripletLoss` for fine-tuning (it pulls the anchor closer to the positive and pushes it away from the negative). The training arguments set epochs, batch size, and warmup steps.

In [ ]:
# Load base model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Loss
train_loss = TripletLoss(model=model)

# Training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="bge-small-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    warmup_steps=50,
    
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4432.64it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 6: Training the Model

Creating a `SentenceTransformerTrainer` with the model, args, train dataset, and loss, then calling `.train()` to start fine-tuning. After training finishes, the fine-tuned model is saved to the `tufan-bge-small-finetuned` folder.

In [174]:

# Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
)

trainer.train()
model.save("tufan-bge-small-finetuned")

c:\Users\kuntal.chakraborty\Desktop\Study-1\ktl_gen\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.51it/s]


## Step 7: Loading the Fine-Tuned Model

Reloading the just-saved fine-tuned model from disk so we can run inference/testing on it.

In [176]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("tufan-bge-small-finetuned")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4166.96it/s]


## Step 8: Converting Sentences into Embeddings

Encoding a few sample sentences with the model to get their embeddings (vector representations). The output shape `(3, 384)` means 3 sentences, each a 384-dimensional vector.

In [177]:
sentences = [
    "What is the return policy?",
    "You can return items within 30 days.",
    "Store timings are 10 AM to 9 PM."
]

embeddings = model.encode(sentences)
print(embeddings.shape)   # (3, 384) — 3 sentences, 384-dim each

(3, 384)


## Step 9: Finding the Relevant Document with Cosine Similarity

Getting embeddings for a query and a set of documents, then calculating similarity scores with `cos_sim` — the document with the highest score is considered most relevant to the query.

In [178]:
from sentence_transformers.util import cos_sim
docs=[
    "Apple company.",
    "Apple is a good fuit, i want to eat it",
]
query_embedding = model.encode("Apple is the best product")
doc_embeddings = model.encode(docs)

scores = cos_sim(query_embedding, doc_embeddings)
print(scores)   # highest score = most relevant doc

tensor([[0.7848, 0.7898]])


## Step 10: Extracting the Most Relevant Document

Computing similarity scores between the query embedding and doc embeddings, then using `argmax` to find the document with the highest (most relevant) score.

In [179]:


doc_embeddings = model.encode(docs)
query_embedding = model.encode("How can I return a product?")

scores = cos_sim(query_embedding, doc_embeddings)

top_result_idx = torch.argmax(scores)
print(f"Most relevant doc: {docs[top_result_idx]}")

Most relevant doc: Apple company.


## Step 11: Comparing Base vs Fine-Tuned Model

Loading both the base model and the fine-tuned model, then comparing their triplet cosine accuracy using the `evaluator` (TripletEvaluator) — to see how much fine-tuning improved the model.

In [182]:
base_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
finetuned_model = SentenceTransformer("tufan-bge-small-finetuned")

print("Base model:", evaluator(base_model))
print("Fine-tuned model:", evaluator(finetuned_model))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5398.88it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6367.27it/s]


Base model: {'triplet-eval-clean_cosine_accuracy': 0.9800000190734863}
Fine-tuned model: {'triplet-eval-clean_cosine_accuracy': 0.9800000190734863}
